In [1]:
# Upload datasets as .csv files in your colab session, and then run this cell.
# Dataset Link: https://drive.google.com/drive/folders/1PzURxyRaN5rclRcyRSI9G8oa1XVSo_SF?usp=drive_link

# Don't work in this colab. Make a copy in your drive instead.

import pandas as pd

copd_df = '/content/COPD_data_preprocessed.csv'
alzheimers_df = '/content/alzheimers_disease_data_preprocessed.csv'
arrhythmia_df = '/content/arrhythmia_data_preprocessed.csv'
autism_df = '/content/autism_screening_data_preprocessed.csv'
diabetes_df = '/content/diabetes_data_preprocessed.csv'
stroke_df = '/content/healthcare-dataset-stroke-data_preprocessed.csv'
heart_df = '/content/heart_disease_uci_preprocessed.csv'
hepatitis_df = '/content/hepatitis_data_preprocessed.csv'
liver_df = '/content/liver_patient_dataset_preprocessed.csv'
thyroid_df = '/content/thyroid_dataset_preprocessed.csv'

In [2]:
# Cell 2: Install TabDPT v1.3 and benchmark dependencies
!pip install --quiet "tabdpt==1.3.*" scikit-learn statsmodels scipy tqdm pandas numpy

# Modeling goes here..

In [3]:
# Cell 3: TabDPT Evaluation Function (Fixed for Colab T4 GPU Attention)
import gc
import os
import warnings
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
from tabdpt import TabDPTClassifier
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# -------------------------------------------------------------
# FIX: Enable Math & Memory Efficient SDPA for Nvidia T4 GPU
# -------------------------------------------------------------
if torch.cuda.is_available():
    try:
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(True)
        torch.backends.cuda.enable_math_sdp(True)
    except Exception:
        pass

# Safe Attention wrapper to prevent "No available kernel" on T4
_orig_sdpa = F.scaled_dot_product_attention


def _safe_sdpa(
    q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None
):
    try:
        return _orig_sdpa(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=dropout_p,
            is_causal=is_causal,
            scale=scale,
        )
    except RuntimeError:
        # Exact mathematical equivalent if T4 kernel is unavailable
        d_k = q.size(-1)
        scaling = scale if scale is not None else (1.0 / (d_k**0.5))
        scores = torch.matmul(q, k.transpose(-2, -1)) * scaling
        if attn_mask is not None:
            scores = scores + attn_mask
        attn_probs = torch.softmax(scores, dim=-1)
        if dropout_p > 0.0:
            attn_probs = torch.dropout(attn_probs, p=dropout_p, train=False)
        return torch.matmul(attn_probs, v)


F.scaled_dot_product_attention = _safe_sdpa

# -------------------------------------------------------------
# 1. Output Format Schema
# -------------------------------------------------------------
TARGET_COLUMNS = [
    "Dataset",
    "Train_N",
    "Test_N",
    "Num_Feats",
    "Domain",
    "DI_mean",
    "DI_sd",
    "EOD_mean",
    "EOD_sd",
    "DPD_mean",
    "DPD_sd",
    "EqOD_mean",
    "EqOD_sd",
    "PPD_mean",
    "PPD_sd",
    "AUROC_mean",
    "AUROC_sd",
    "Sensitivity_mean",
    "Sensitivity_sd",
    "Specificity_mean",
    "Specificity_sd",
    "PPV_mean",
    "PPV_sd",
    "NPV_mean",
    "NPV_sd",
    "ECE_mean",
    "ECE_sd",
    "NLL_mean",
    "NLL_sd",
    "Brier_mean",
    "Brier_sd",
    "Cal_Slope_mean",
    "Cal_Slope_sd",
    "Cal_Intercept_mean",
    "Cal_Intercept_sd",
    "AUROC_Deg_mean",
    "AUROC_Deg_sd",
    "Accuracy_Deg_mean",
    "Accuracy_Deg_sd",
    "F1_Deg_mean",
    "F1_Deg_sd",
    "Flip_Rate_mean",
    "Flip_Rate_sd",
    "MAPC_mean",
    "MAPC_sd",
]


def run_tabdpt_dataset(name, path, domain, output_csv="TabDPT_Results.csv"):
    if not os.path.exists(path):
        print(f"Error: {path} not found.")
        return

    print(f"\n==========================================")
    print(f"Running TabDPT on: {name} ({domain})")
    print(f"==========================================")

    df = pd.read_csv(path)
    target_col = df.columns[-1]
    sex_col_name = df.columns[0]
    feature_cols = df.columns[:-1].tolist()
    num_feats = len(feature_cols)

    X = df[feature_cols].copy()
    y = df[target_col].copy().to_numpy().astype(int)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_metrics = []
    train_n_list = []
    test_n_list = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        tqdm(skf.split(X, y), total=5, desc=f"{name} 5-Folds")
    ):
        train_n_list.append(len(train_idx))
        test_n_list.append(len(test_idx))

        X_train_df, y_train = X.iloc[train_idx].copy(), y[train_idx]
        X_test_df, y_test = X.iloc[test_idx].copy(), y[test_idx]

        imputer = KNNImputer(n_neighbors=5)
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(X_train_df), columns=feature_cols
        )
        X_test_imp = pd.DataFrame(imputer.transform(X_test_df), columns=feature_cols)
        sex_test = X_test_imp[sex_col_name].to_numpy().astype(int)

        # TabDPT model on GPU
        model = TabDPTClassifier(device="cuda")
        model.fit(X_train_imp.values, y_train)

        def get_probs(m, x_data):
            pr = m.predict_proba(x_data)
            if hasattr(pr, "values"):
                pr = pr.values
            pr = np.asarray(pr)
            return pr[:, 1] if pr.ndim == 2 else pr

        # Clean Test Predictions
        p_clean = get_probs(model, X_test_imp.values)
        y_clean_pred = (p_clean >= 0.5).astype(int)

        # Robustness Noise (10% standard deviation, excluding sex and outcome)
        X_test_noisy = X_test_imp.copy()
        np.random.seed(42 + fold_idx)
        for col in feature_cols[1:]:
            col_std = np.std(X_train_imp[col].values, ddof=1)
            noise_scale = 0.10 * (col_std if (not np.isnan(col_std) and col_std > 0) else 1.0)
            X_test_noisy[col] += np.random.normal(0, noise_scale, size=len(X_test_noisy))

        # Noisy Predictions
        p_noisy = get_probs(model, X_test_noisy.values)
        y_noisy_pred = (p_noisy >= 0.5).astype(int)

        # 1. Fairness
        rates = {}
        for grp, grp_name in [(0, "F"), (1, "M")]:
            mask = sex_test == grp
            yt, yp = y_test[mask], y_clean_pred[mask]
            tp = np.sum((yt == 1) & (yp == 1))
            fn = np.sum((yt == 1) & (yp == 0))
            fp = np.sum((yt == 0) & (yp == 1))
            tn = np.sum((yt == 0) & (yp == 0))
            rates[grp_name] = {
                "SR": np.mean(yp == 1) if len(yp) > 0 else np.nan,
                "TPR": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
                "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
                "PPV": tp / (tp + fp) if (tp + fp) > 0 else np.nan,
            }

        s_m, s_f = rates["M"]["SR"], rates["F"]["SR"]
        t_m, t_f = rates["M"]["TPR"], rates["F"]["TPR"]
        f_m, f_f = rates["M"]["FPR"], rates["F"]["FPR"]
        p_m, p_f = rates["M"]["PPV"], rates["F"]["PPV"]

        di = (
            np.nan
            if (np.isnan(s_m) or np.isnan(s_f) or max(s_m, s_f) == 0)
            else min(s_m, s_f) / max(s_m, s_f)
        )
        eod = np.nan if (np.isnan(t_m) or np.isnan(t_f)) else abs(t_m - t_f)
        dpd = np.nan if (np.isnan(s_m) or np.isnan(s_f)) else abs(s_m - s_f)
        eqod = (
            np.nan
            if (np.isnan(t_m) or np.isnan(t_f) or np.isnan(f_m) or np.isnan(f_f))
            else max(abs(t_m - t_f), abs(f_m - f_f))
        )
        ppd = np.nan if (np.isnan(p_m) or np.isnan(p_f)) else abs(p_m - p_f)

        # 2. Predictive Discrimination
        tp = np.sum((y_test == 1) & (y_clean_pred == 1))
        tn = np.sum((y_test == 0) & (y_clean_pred == 0))
        fp = np.sum((y_test == 0) & (y_clean_pred == 1))
        fn = np.sum((y_test == 1) & (y_clean_pred == 0))
        try:
            auroc = roc_auc_score(y_test, p_clean)
        except Exception:
            auroc = np.nan

        # 3. Uncertainty / Calibration
        p_c = np.clip(p_clean, 1e-15, 1.0 - 1e-15)
        bins = np.linspace(0.0, 1.0, 16)
        ece = sum(
            (np.sum((p_c >= bins[b]) & (p_c <= bins[b + 1])) / len(y_test))
            * abs(
                np.mean(y_test[(p_c >= bins[b]) & (p_c <= bins[b + 1])])
                - np.mean(p_c[(p_c >= bins[b]) & (p_c <= bins[b + 1])])
            )
            for b in range(15)
            if np.sum((p_c >= bins[b]) & (p_c <= bins[b + 1])) > 0
        )
        nll = -np.mean(y_test * np.log(p_c) + (1 - y_test) * np.log(1 - p_c))
        brier = np.mean((p_c - y_test) ** 2)
        try:
            m_cal = sm.Logit(y_test, sm.add_constant(logit(p_c))).fit(
                disp=False, maxiter=100
            )
            c_int, c_slope = m_cal.params[0], m_cal.params[1]
        except Exception:
            c_int, c_slope = np.nan, np.nan

        # 4. Robustness
        try:
            auroc_deg = auroc - roc_auc_score(y_test, p_noisy)
        except Exception:
            auroc_deg = np.nan
        acc_deg = accuracy_score(y_test, y_clean_pred) - accuracy_score(y_test, y_noisy_pred)
        f1_deg = f1_score(y_test, y_clean_pred, zero_division=0) - f1_score(
            y_test, y_noisy_pred, zero_division=0
        )
        flip = np.mean(y_clean_pred != y_noisy_pred)
        mapc = np.mean(np.abs(p_clean - p_noisy))

        fold_metrics.append({
            "DI": di, "EOD": eod, "DPD": dpd, "EqOD": eqod, "PPD": ppd,
            "AUROC": auroc,
            "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
            "Specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
            "PPV": tp / (tp + fp) if (tp + fp) > 0 else np.nan,
            "NPV": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
            "ECE": ece, "NLL": nll, "Brier": brier, "Cal_Slope": c_slope, "Cal_Intercept": c_int,
            "AUROC_Deg": auroc_deg, "Accuracy_Deg": acc_deg, "F1_Deg": f1_deg,
            "Flip_Rate": flip, "MAPC": mapc,
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    f_df = pd.DataFrame(fold_metrics)
    row = {
        "Dataset": name,
        "Train_N": float(np.mean(train_n_list)),
        "Test_N": float(np.mean(test_n_list)),
        "Num_Feats": int(num_feats),
        "Domain": domain,
    }
    for k in f_df.columns:
        row[f"{k}_mean"] = f_df[k].mean()
        row[f"{k}_sd"] = f_df[k].std(ddof=1)

    new_row_df = pd.DataFrame([row]).reindex(columns=TARGET_COLUMNS)

    # Save / Update in TabDPT_Results.csv
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        existing_df = existing_df[existing_df["Dataset"] != name]
        combined_df = pd.concat([existing_df, new_row_df], ignore_index=True)
    else:
        combined_df = new_row_df

    combined_df = combined_df.reindex(columns=TARGET_COLUMNS)
    combined_df.to_csv(output_csv, index=False, float_format="%.6f")

    print(f"Done! {name} recorded in {output_csv}.")
    display(new_row_df)

In [4]:
run_tabdpt_dataset(
    "Alzheimers",
    "/content/alzheimers_disease_data_preprocessed.csv",
    "Neurology",
)


Running TabDPT on: Alzheimers (Neurology)


Alzheimers 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

tabdpt1_3.safetensors: reconstructing file:   0%|          |  0.00B /  252MB            

tabdpt1_3.safetensors: downloading bytes:           |  0.00B            

W1119 11:10:36.773000 374 torch/_inductor/utils.py:1806] [4/0_1] Not enough SMs to use max_autotune_gemm mode


Done! Alzheimers recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Alzheimers,560.0,140.0,32,Neurology,0.786872,0.09138,0.069129,0.036707,0.118692,...,0.002571,0.002604,0.011429,0.019299,0.011678,0.019999,0.034286,0.017053,0.029388,0.00617


In [5]:
run_tabdpt_dataset(
    "Arrhythmia", "/content/arrhythmia_data_preprocessed.csv", "Cardiology"
)


Running TabDPT on: Arrhythmia (Cardiology)


Arrhythmia 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Arrhythmia recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Arrhythmia,361.6,90.4,262,Cardiology,0.635901,0.073738,0.167135,0.06077,0.181334,...,0.004899,0.003443,-0.004444,0.012669,-0.005327,0.012991,0.017778,0.016851,0.016494,0.002185


In [6]:
run_tabdpt_dataset(
    "Autism",
    "/content/autism_screening_data_preprocessed.csv",
    "Psychiatry",
)


Running TabDPT on: Autism (Psychiatry)


Autism 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Autism recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Autism,559.2,139.8,19,Psychiatry,0.711783,0.139244,0.0,0.0,0.094228,...,0.000052,0.000115,0.014306,0.008748,0.025779,0.015711,0.014306,0.008748,0.016185,0.009036


In [7]:
run_tabdpt_dataset("COPD", "/content/COPD_data_preprocessed.csv", "Pulmonology")


Running TabDPT on: COPD (Pulmonology)


COPD 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! COPD recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,COPD,504.0,126.0,14,Pulmonology,0.625561,0.123456,0.089735,0.111915,0.317289,...,0.009979,0.014927,-0.001587,0.014197,-0.000897,0.009741,0.02381,0.015873,0.066033,0.01093


In [ ]:
run_tabdpt_dataset(
    "Diabetes", "/content/diabetes_data_preprocessed.csv", "Endocrinology"
)


Running TabDPT on: Diabetes (Endocrinology)


Diabetes 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Diabetes recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Diabetes,560.0,140.0,8,Endocrinology,0.908859,0.130328,0.044548,0.024051,0.053278,...,0.008245,0.00796,0.017143,0.027939,0.015052,0.027961,0.06,0.021783,0.03759,0.008771


In [8]:
run_tabdpt_dataset(
    "Stroke",
    "/content/healthcare-dataset-stroke-data_preprocessed.csv",
    "Neurology",
)


Running TabDPT on: Stroke (Neurology)


Stroke 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Stroke recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Stroke,560.0,140.0,10,Neurology,0.891022,0.104636,0.10737,0.048641,0.037398,...,0.006581,0.005429,0.008571,0.011737,0.009124,0.021268,0.025714,0.022924,0.02242,0.002287


In [ ]:
run_tabdpt_dataset(
    "Heart Disease",
    "/content/heart_disease_uci_preprocessed.csv",
    "Cardiology",
)


Running TabDPT on: Heart Disease (Cardiology)


Heart Disease 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Heart Disease recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Heart Disease,560.0,140.0,14,Cardiology,0.295723,0.122957,0.204741,0.198721,0.42232,...,0.004592,0.002419,-0.002857,0.009583,-0.004354,0.009625,0.02,0.012778,0.026628,0.005561


In [9]:
run_tabdpt_dataset(
    "Hepatitis",
    "/content/hepatitis_data_preprocessed.csv",
    "Infectious Disease",
)


Running TabDPT on: Hepatitis (Infectious Disease)


Hepatitis 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Hepatitis recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Hepatitis,124.0,31.0,19,Infectious Disease,0.173333,0.387585,NaN,NaN,0.163119,...,-0.001333,0.002981,-0.006452,0.014426,-0.012121,0.027104,0.006452,0.014426,0.011186,0.002248


In [10]:
run_tabdpt_dataset(
    "Liver Disease",
    "/content/liver_patient_dataset_preprocessed.csv",
    "Hepatology",
)


Running TabDPT on: Liver Disease (Hepatology)


Liver Disease 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Liver Disease recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Liver Disease,560.0,140.0,10,Hepatology,0.87851,0.062514,0.065507,0.062401,0.063584,...,0.21151,0.034814,0.295714,0.041219,0.201626,0.028764,0.37,0.053309,0.283366,0.020628


In [11]:
run_tabdpt_dataset(
    "Thyroid", "/content/thyroid_dataset_preprocessed.csv", "Endocrinology"
)


Running TabDPT on: Thyroid (Endocrinology)


Thyroid 5-Folds:   0%|          | 0/5 [00:00<?, ?it/s]

Done! Thyroid recorded in TabDPT_Results.csv.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,Thyroid,560.0,140.0,21,Endocrinology,0.780274,0.127599,0.003704,0.008282,0.121756,...,0.115735,0.041064,0.172857,0.05565,0.171045,0.053559,0.184286,0.045006,0.222448,0.040031


In [12]:
#Sort all 10 datasets and download TabDPT_Results.csv
import pandas as pd
from google.colab import files

CSV_PATH = "TabDPT_Results.csv"
df = pd.read_csv(CSV_PATH)

standard_order = [
    "COPD",
    "Alzheimers",
    "Arrhythmia",
    "Autism",
    "Diabetes",
    "Stroke",
    "Heart Disease",
    "Hepatitis",
    "Liver Disease",
    "Thyroid",
]

df["Dataset"] = pd.Categorical(
    df["Dataset"], categories=standard_order, ordered=True
)
df = df.sort_values("Dataset").reset_index(drop=True)
df.to_csv(CSV_PATH, index=False, float_format="%.6f")

print(f"All {len(df)}/10 datasets confirmed and saved!")
display(df[["Dataset", "Domain", "Train_N", "Test_N", "AUROC_mean"]])

# Trigger download
files.download(CSV_PATH)

All 8/10 datasets confirmed and saved!


,Dataset,Domain,Train_N,Test_N,AUROC_mean
0,COPD,Pulmonology,504.0,126.0,0.918121
1,Alzheimers,Neurology,560.0,140.0,0.951918
2,Arrhythmia,Cardiology,361.6,90.4,0.827796
3,Autism,Psychiatry,559.2,139.8,1.000000
4,Stroke,Neurology,560.0,140.0,0.835836
5,Hepatitis,Infectious Disease,124.0,31.0,0.848952
6,Liver Disease,Hepatology,560.0,140.0,0.963694
7,Thyroid,Endocrinology,560.0,140.0,0.995755


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>